In [5]:
import pandas as pd

anno_path = "GPL6480-9577.txt"
data_path = "哮喘靶点_Lasso+XGBoost.csv"
save_path = "探针转基因结果.csv"

anno = pd.read_csv(anno_path, sep="\t", skiprows=17, dtype=str)

anno_map = anno[["ID", "GENE_SYMBOL"]].drop_duplicates()
anno_map.columns = ["Gene", "Gene_Symbol"]

# 读取数据表
df = pd.read_csv(data_path)

# 左连接匹配基因名
res = pd.merge(df, anno_map, on="Gene", how="left")

# 保存
res.to_csv(save_path, index=False, encoding="utf-8-sig")

print("转换完成！")
print(f"总共 {len(res)} 个探针")
print(f"成功匹配基因名：{res['Gene_Symbol'].notna().sum()} 个")

转换完成！
总共 130 个探针
成功匹配基因名：130 个


In [2]:
import pandas as pd

compound_file = "成分-靶点列表.xlsx"
gene_file = "探针转基因结果.csv"
output_file = "药物-疾病基因交集结果.csv"

# 1. 读取成分-靶点文件
df_compound = pd.read_excel(compound_file)
df_compound.columns = ["成分", "Common_name"]

# 2. 读取疾病重要基因
df_gene = pd.read_csv(gene_file)

# 3. 提取基因列表
gene_list1 = df_compound["Common_name"].dropna().astype(str).str.strip().tolist()
gene_list2 = df_gene["Gene_Symbol"].dropna().astype(str).str.strip().tolist()

# 4. 计算基因交集
set1 = set(gene_list1)
set2 = set(gene_list2)
common_genes = sorted(set1 & set2)

# 5. 控制台输出核心结果
print("="*60)
print(f"找到交集基因数量：{len(common_genes)}")
print(f"交集基因列表：{common_genes}")
print("="*60)

# 6. 匹配完整信息
result_df = df_compound[df_compound["Common_name"].isin(common_genes)]
result_df = result_df.merge(
    df_gene,
    left_on="Common_name",
    right_on="Gene_Symbol",
    how="left"
)
# 整理列顺序，只保留核心列
result_df = result_df[["成分", "Common_name", "Importance"]].sort_values(
    by="Importance",
    ascending=False
)

result_df.to_csv(output_file, index=False, encoding="utf-8-sig")

# 8. 输出结果预览
print(f"\n分析完成！结果已保存到：{output_file}")
print("\n结果预览（前10行）：")
print(result_df.head(10))

找到交集基因数量：4
交集基因列表：['CDC7', 'FGFR1', 'KIT', 'MMP8']

分析完成！结果已保存到：药物-疾病基因交集结果.csv

结果预览（前10行）：
                        成分 Common_name  Importance
8                   Ononin         KIT    0.043296
6           Liquiritigenin         KIT    0.043296
10    3-O-Angeloylhamaudol         KIT    0.043296
2            Neoliquiritin        MMP8    0.000000
0               Albiflorin        MMP8    0.000000
1               Albiflorin       FGFR1    0.000000
5   5-O-Methylvisammioside       FGFR1    0.000000
4                Cimifugin        MMP8    0.000000
3               Liquiritin        MMP8    0.000000
7           Liquiritigenin       FGFR1    0.000000
